In [1]:
import sys
import os

# Clear any previously loaded mmengine/bitsandbytes/numpy/matplotlib/pycocotools modules so the notebook uses the active env install
for key in list(sys.modules):
    if key in ('mmengine', 'bitsandbytes', 'numpy', 'matplotlib', 'pycocotools') or key.startswith(('mmengine.', 'bitsandbytes.', 'numpy.', 'matplotlib.', 'pycocotools.')):
        sys.modules.pop(key)

import torch
import mmcv
import mmdet
import mmengine
import numpy as np
import matplotlib as mpl
import pycocotools

print("Python executable:", sys.executable)
print("sys.path first entries:", sys.path[:6])
print("Torch version:", torch.__version__)
print("MMCV version:", mmcv.__version__)
print("MMDet version:", mmdet.__version__)
print("MMEngine version:", mmengine.__version__)
print("MMEngine path:", mmengine.__file__)
print("NumPy path:", np.__file__)
print("Matplotlib path:", mpl.__file__)
print("Pycocotools path:", pycocotools.__file__)
print("Matplotlib rcParams present:", hasattr(mpl, 'rcParams'))
print("CUDA available:", torch.cuda.is_available())

Python executable: /home/ug/.conda/envs/camouflage/bin/python
sys.path first entries: ['/home/ug/projects/codec-robust-deepfake-detection', '/home/ug/.conda/envs/camouflage/lib/python310.zip', '/home/ug/.conda/envs/camouflage/lib/python3.10', '/home/ug/.conda/envs/camouflage/lib/python3.10/lib-dynload', '', '/home/ug/.local/lib/python3.10/site-packages']
Torch version: 2.6.0+cu124
MMCV version: 2.0.1
MMDet version: 3.1.0
MMEngine version: 0.10.7
MMEngine path: /home/ug/.conda/envs/camouflage/lib/python3.10/site-packages/mmengine/__init__.py
NumPy path: /home/ug/.conda/envs/camouflage/lib/python3.10/site-packages/numpy/__init__.py
Matplotlib path: /home/ug/.conda/envs/camouflage/lib/python3.10/site-packages/matplotlib/__init__.py
Pycocotools path: /home/ug/.conda/envs/camouflage/lib/python3.10/site-packages/pycocotools/__init__.py
Matplotlib rcParams present: True
CUDA available: True


In [2]:
from pathlib import Path
from mmengine.config import Config
from mmengine.runner import Runner

import os

[2026-06-08 05:49:29,708] [INFO] [real_accelerator.py:158:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/home/ug/.local/lib/python3.10/site-packages/deepspeed/runtime/zero/linear.py:49: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  def forward(ctx, input, weight, bias=None):
/home/ug/.local/lib/python3.10/site-packages/deepspeed/runtime/zero/linear.py:67: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  def backward(ctx, grad_output):


In [3]:
DATA_ROOT = "/data/UG/Kiranmoy/datasets"

TRAIN_JSON = f"{DATA_ROOT}/MHCD2022_COCO_Labels/train_coco.json"
VAL_JSON   = f"{DATA_ROOT}/MHCD2022_COCO_Labels/val_coco.json"
TEST_JSON  = f"{DATA_ROOT}/MHCD2022_COCO_Labels/test_coco.json"

TRAIN_IMG = f"{DATA_ROOT}/MHCD2022_YOLO/images/train"
VAL_IMG   = f"{DATA_ROOT}/MHCD2022_YOLO/images/val"
TEST_IMG  = f"{DATA_ROOT}/MHCD2022_YOLO/images/test"

WORK_DIR = "./work_dirs/mhcd2022_faster_rcnn"

In [4]:
from mmdet.utils import register_all_modules
register_all_modules()

cfg = Config.fromfile(
    str(Path(mmdet.__file__).parent / ".mim" / "configs" / "faster_rcnn" / "faster-rcnn_r50_fpn_1x_coco.py")
)

print(cfg.model.type)


FasterRCNN


In [5]:
classes = (
    "person",
    "military vehicle",
    "tank",
    "aeroplane",
    "warship"
)

NUM_CLASSES = len(classes)

cfg.work_dir = WORK_DIR

In [6]:
cfg.dataset_type = "CocoDataset"

cfg.train_dataloader.dataset.metainfo = dict(classes=classes)
cfg.val_dataloader.dataset.metainfo   = dict(classes=classes)
cfg.test_dataloader.dataset.metainfo  = dict(classes=classes)

cfg.train_dataloader.dataset.data_root = ""
cfg.val_dataloader.dataset.data_root   = ""
cfg.test_dataloader.dataset.data_root  = ""

cfg.train_dataloader.dataset.ann_file = TRAIN_JSON
cfg.val_dataloader.dataset.ann_file   = VAL_JSON
cfg.test_dataloader.dataset.ann_file  = TEST_JSON

cfg.train_dataloader.dataset.data_prefix = dict(img=TRAIN_IMG + "/")
cfg.val_dataloader.dataset.data_prefix   = dict(img=VAL_IMG + "/")
cfg.test_dataloader.dataset.data_prefix  = dict(img=TEST_IMG + "/")

In [7]:
cfg.model.roi_head.bbox_head.num_classes = NUM_CLASSES

In [8]:
cfg.train_cfg.max_epochs = 50

cfg.default_hooks.checkpoint.interval = 1

cfg.train_dataloader.batch_size = 8
cfg.train_dataloader.num_workers = 8

cfg.val_dataloader.batch_size = 4
cfg.test_dataloader.batch_size = 4

cfg.seed = 42

cfg.load_from = None
print("Training from scratch: cfg.load_from = None")

Training from scratch: cfg.load_from = None


In [9]:
cfg.val_evaluator.ann_file = VAL_JSON
cfg.test_evaluator.ann_file = TEST_JSON

In [10]:
cfg.dump("mhcd2022_faster_rcnn.py")

print("Config saved.")

Config saved.


In [11]:
print(cfg.model.type)
print(cfg.model.roi_head.bbox_head.num_classes)

FasterRCNN
5


In [12]:
print(cfg.train_dataloader.dataset.ann_file)
print(cfg.val_dataloader.dataset.ann_file)
print(cfg.test_dataloader.dataset.ann_file)

/data/UG/Kiranmoy/datasets/MHCD2022_COCO_Labels/train_coco.json
/data/UG/Kiranmoy/datasets/MHCD2022_COCO_Labels/val_coco.json
/data/UG/Kiranmoy/datasets/MHCD2022_COCO_Labels/test_coco.json


In [13]:
print(cfg.load_from)

None


In [14]:
from mmdet.utils import register_all_modules
register_all_modules(init_default_scope=True)
cfg.default_scope = 'mmdet'

# Train from scratch without MMEngine's visualizer, which is failing in this environment.
cfg.visualizer = None

if cfg.get('visualizer') is not None:
    cfg.visualizer.type = 'Visualizer'

# Ensure dataloader persistent_workers and num_workers are compatible
for dl_key in ('train_dataloader','val_dataloader','test_dataloader'):
    dl = cfg.get(dl_key)
    if isinstance(dl, dict):
        if dl.get('persistent_workers', False) and dl.get('num_workers', 0) == 0:
            dl['persistent_workers'] = False
        if dl.get('persistent_workers', False) and dl.get('num_workers', 0) == 0:
            dl['num_workers'] = 1

# Fallback: ensure train_dataloader has num_workers >= 1 when persistent_workers True
if cfg.get('train_dataloader') and cfg.train_dataloader.get('persistent_workers', False) and cfg.train_dataloader.get('num_workers', 0) == 0:
    cfg.train_dataloader['num_workers'] = 1

runner = Runner.from_cfg(cfg)
runner.train()

06/08 05:49:32 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.10.20 (main, Mar 11 2026, 17:46:40) [GCC 14.3.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 1918926977
    GPU 0: NVIDIA H100 NVL
    CUDA_HOME: /usr/local/cuda-12.4
    NVCC: Cuda compilation tools, release 12.4, V12.4.99
    GCC: gcc (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0
    PyTorch: 2.6.0+cu124
    PyTorch compiling details: PyTorch built with:
  - GCC 9.3
  - C++ Version: 201703
  - Intel(R) oneAPI Math Kernel Library Version 2024.2-Product Build 20240605 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v3.5.3 (Git Hash 66f0cb9eb66affd2da3bf5f8d897376f04aae6af)
  - OpenMP 201511 (a.k.a. OpenMP 4.5)
  - LAPACK is enabled (usually provided by MKL)
  - NNPACK is enabled
  - CPU capability usage: AVX512
  - CUDA Runtime 12.4
  - NVCC architecture flags: -gencode;arch=compute_50,code=sm_5

OutOfMemoryError: CUDA out of memory. Tried to allocate 862.00 MiB. GPU 0 has a total capacity of 93.00 GiB of which 547.00 MiB is free. Process 929354 has 76.88 GiB memory in use. Including non-PyTorch memory, this process has 15.48 GiB memory in use. Of the allocated memory 12.99 GiB is allocated by PyTorch, and 1.75 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
work_dir = Path("./work_dirs/mhcd2022_faster_rcnn")

for f in sorted(work_dir.glob("*.pth")):
    print(f.name)

epoch_1.pth
epoch_10.pth
epoch_11.pth
epoch_12.pth
epoch_13.pth
epoch_14.pth
epoch_15.pth
epoch_16.pth
epoch_17.pth
epoch_18.pth
epoch_19.pth
epoch_2.pth
epoch_20.pth
epoch_21.pth
epoch_22.pth
epoch_23.pth
epoch_24.pth
epoch_25.pth
epoch_26.pth
epoch_27.pth
epoch_28.pth
epoch_29.pth
epoch_3.pth
epoch_30.pth
epoch_31.pth
epoch_32.pth
epoch_33.pth
epoch_34.pth
epoch_35.pth
epoch_36.pth
epoch_37.pth
epoch_38.pth
epoch_39.pth
epoch_4.pth
epoch_40.pth
epoch_41.pth
epoch_42.pth
epoch_43.pth
epoch_44.pth
epoch_45.pth
epoch_46.pth
epoch_47.pth
epoch_48.pth
epoch_49.pth
epoch_5.pth
epoch_50.pth
epoch_6.pth
epoch_7.pth
epoch_8.pth
epoch_9.pth


In [ ]:
CHECKPOINT = (
    "./work_dirs/mhcd2022_faster_rcnn/epoch_50.pth"
)

In [ ]:
import torch
from mmengine.logging.history_buffer import HistoryBuffer

torch.serialization.add_safe_globals([HistoryBuffer])

print("Safe globals added")

Safe globals added


In [ ]:
from mmengine.runner import Runner

runner = Runner.from_cfg(cfg)

06/08 05:43:52 - mmengine - WARNING - Failed to search registry with scope "mmdet" in the "log_processor" registry tree. As a workaround, the current "log_processor" registry in "mmengine" is used to build instance. This may cause unexpected failure when running the built modules. Please check whether "mmdet" is a correct scope, or whether the registry is initialized.
06/08 05:43:52 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.10.20 (main, Mar 11 2026, 17:46:40) [GCC 14.3.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 2020487299
    GPU 0: NVIDIA H100 NVL
    CUDA_HOME: /usr/local/cuda-12.4
    NVCC: Cuda compilation tools, release 12.4, V12.4.99
    GCC: gcc (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0
    PyTorch: 2.6.0+cu124
    PyTorch compiling details: PyTorch built with:
  - GCC 9.3
  - C++ Version: 201703
  - Intel(R) oneAPI Math Kernel Library Version 2024.2-

TypeError: float() argument must be a string or a real number, not '_NoValueType'

In [ ]:
import torch

ckpt_path = "./work_dirs/mhcd2022_faster_rcnn/epoch_50.pth"

checkpoint = torch.load(
    ckpt_path,
    map_location="cpu",
    weights_only=False  # important
)

runner.model.load_state_dict(checkpoint["state_dict"])

print("Checkpoint loaded successfully!")

NameError: name 'runner' is not defined

In [ ]:
from mmdet.utils import register_all_modules
register_all_modules(init_default_scope=True)
cfg.default_scope = 'mmdet'

# This notebook has already trained a model in work_dirs; load that trained checkpoint for evaluation.
cfg.load_from = None
cfg.resume = False

if cfg.get('visualizer') is not None:
    cfg.visualizer.type = 'Visualizer'

# Ensure dataloader persistent_workers and num_workers are compatible
for dl_key in ('train_dataloader', 'val_dataloader', 'test_dataloader'):
    dl = cfg.get(dl_key)
    if isinstance(dl, dict):
        if dl.get('persistent_workers', False) and dl.get('num_workers', 0) == 0:
            dl['persistent_workers'] = False
        if dl.get('persistent_workers', False) and dl.get('num_workers', 0) == 0:
            dl['num_workers'] = 1

# Fallback: ensure train_dataloader has num_workers >= 1 when persistent_workers True
if cfg.get('train_dataloader') and cfg.train_dataloader.get('persistent_workers', False) and cfg.train_dataloader.get('num_workers', 0) == 0:
    cfg.train_dataloader['num_workers'] = 1

runner = Runner.from_cfg(cfg)

ckpt_path = Path(WORK_DIR) / 'epoch_50.pth'
checkpoint = torch.load(ckpt_path, map_location='cpu', weights_only=False)
runner.model.load_state_dict(checkpoint['state_dict'])
print(f'Loaded trained checkpoint: {ckpt_path}')

metrics = runner.test()
print(metrics)

06/08 05:05:10 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.10.20 (main, Mar 11 2026, 17:46:40) [GCC 14.3.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 1502483054
    GPU 0: NVIDIA H100 NVL
    CUDA_HOME: /usr/local/cuda-12.4
    NVCC: Cuda compilation tools, release 12.4, V12.4.99
    GCC: gcc (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0
    PyTorch: 2.6.0+cu124
    PyTorch compiling details: PyTorch built with:
  - GCC 9.3
  - C++ Version: 201703
  - Intel(R) oneAPI Math Kernel Library Version 2024.2-Product Build 20240605 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v3.5.3 (Git Hash 66f0cb9eb66affd2da3bf5f8d897376f04aae6af)
  - OpenMP 201511 (a.k.a. OpenMP 4.5)
  - LAPACK is enabled (usually provided by MKL)
  - NNPACK is enabled
  - CPU capability usage: AVX512
  - CUDA Runtime 12.4
  - NVCC architecture flags: -gencode;arch=compute_50,code=sm_5

06/08 05:05:10 - mmengine - INFO - Config:
auto_scale_lr = dict(base_batch_size=16, enable=False)
backend_args = None
data_root = 'data/coco/'
dataset_type = 'CocoDataset'
default_hooks = dict(
    checkpoint=dict(interval=1, type='CheckpointHook'),
    logger=dict(interval=50, type='LoggerHook'),
    param_scheduler=dict(type='ParamSchedulerHook'),
    sampler_seed=dict(type='DistSamplerSeedHook'),
    timer=dict(type='IterTimerHook'),
    visualization=dict(type='DetVisualizationHook'))
default_scope = 'mmdet'
env_cfg = dict(
    cudnn_benchmark=False,
    dist_cfg=dict(backend='nccl'),
    mp_cfg=dict(mp_start_method='fork', opencv_num_threads=0))
load_from = None
log_level = 'INFO'
log_processor = dict(by_epoch=True, type='LogProcessor', window_size=50)
model = dict(
    backbone=dict(
        depth=50,
        frozen_stages=1,
        init_cfg=dict(checkpoint='torchvision://resnet50', type='Pretrained'),
        norm_cfg=dict(requires_grad=True, type='BN'),
        norm_eval=True,

/home/ug/.conda/envs/camouflage/lib/python3.10/site-packages/mmengine/utils/manager.py:113: UserWarning: <class 'mmengine.visualization.visualizer.Visualizer'> instance named of visualizer has been created, the method `get_instance` should not accept any other arguments
  warnings.warn(


06/08 05:05:11 - mmengine - INFO - Distributed training is not used, all SyncBatchNorm (SyncBN) layers in the model will be automatically reverted to BatchNormXd layers if they are used.
06/08 05:05:11 - mmengine - INFO - Hooks will be executed in the following order:
before_run:
(VERY_HIGH   ) RuntimeInfoHook                    
(BELOW_NORMAL) LoggerHook                         
 -------------------- 
before_train:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(VERY_LOW    ) CheckpointHook                     
 -------------------- 
before_train_epoch:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(NORMAL      ) DistSamplerSeedHook                
 -------------------- 
before_train_iter:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
 -------------------- 
after_train_iter:
(VERY_HIGH   ) RuntimeInfoHook                

In [ ]:
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import json
import numpy as np
import pandas as pd

In [ ]:
GT_JSON = "/data/UG/Kiranmoy/datasets/MHCD2022_COCO_Labels/test_coco.json"

PRED_JSON = "./faster_rcnn_test_predictions.json"

In [ ]:
from pathlib import Path
import torch
import mmdet
from mmengine.config import Config
from numpy.core.multiarray import _reconstruct
from mmdet.utils import register_all_modules
from mmdet.apis import DetInferencer

# Make this cell self-contained so it can be rerun independently.
register_all_modules(init_default_scope=True)
config_path = Path(mmdet.__file__).parent / ".mim" / "configs" / "faster_rcnn" / "faster-rcnn_r50_fpn_1x_coco.py"
cfg = Config.fromfile(str(config_path))
cfg.default_scope = 'mmdet'

# Allow PyTorch 2.6 safe-loading for this trusted checkpoint.
torch.serialization.add_safe_globals([_reconstruct])

inferencer = DetInferencer(
    model=cfg,
    weights="./work_dirs/mhcd2022_faster_rcnn/epoch_50.pth",
    device="cuda:0"
)

TypeError: model must be a filepath or any ConfigTypeobject, but got <class 'mmengine.config.config.Config'>